In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_log_error

DATA_DIR = "../data"

train = pd.read_parquet(f"{DATA_DIR}/train_processed.parquet")
test = pd.read_parquet(f"{DATA_DIR}/test_processed.parquet")

print("train:", train.shape)
print("test:", test.shape)

train: (3000888, 27)
test: (28512, 27)


In [2]:
train = train.sort_values("date")

cutoff_date = train["date"].max() - pd.Timedelta(days=16)

train_part = train[train["date"] <= cutoff_date]
val_part = train[train["date"] > cutoff_date]

print("Train part:", train_part.shape, train_part["date"].max())
print("Val part:", val_part.shape, val_part["date"].min(), "-", val_part["date"].max())

Train part: (2972376, 27) 2017-07-30 00:00:00
Val part: (28512, 27) 2017-07-31 00:00:00 - 2017-08-15 00:00:00


In [3]:
val_part = val_part.copy()
val_part["prediction_naive"] = val_part["lag_7"]

# Handle any missing lag values (shouldn't be many at this point in the series)
print("Missing lag_7 in val_part:", val_part["prediction_naive"].isna().sum())

val_part["prediction_naive"] = val_part["prediction_naive"].fillna(0)
val_part["prediction_naive"] = val_part["prediction_naive"].clip(lower=0)

Missing lag_7 in val_part: 0


In [4]:
rmsle_naive = np.sqrt(mean_squared_log_error(val_part["sales"], val_part["prediction_naive"]))
print(f"Naive baseline (lag_7) RMSLE: {rmsle_naive:.4f}")

Naive baseline (lag_7) RMSLE: 0.5694


In [5]:
# columns that should NOT be used as features
exclude_cols = ["id", "date", "sales", "sales_log", "is_train"]

feature_cols = [col for col in train_part.columns if col not in exclude_cols]

print("Number of features:", len(feature_cols))
print(feature_cols)

Number of features: 22
['store_nbr', 'family', 'onpromotion', 'day_of_week', 'day_of_month', 'month', 'year', 'is_weekend', 'is_holiday', 'is_work_day', 'oil_price', 'city', 'state', 'type', 'cluster', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_std_7', 'rolling_mean_28', 'rolling_std_28']


In [6]:
categorical_cols = ["store_nbr", "family", "city", "state", "type", "cluster"]

for col in categorical_cols:
    train_part[col] = train_part[col].astype("category")
    val_part[col] = val_part[col].astype("category")

print(train_part[categorical_cols].dtypes)

store_nbr    category
family       category
city         category
state        category
type         category
cluster      category
dtype: object


In [7]:
import lightgbm as lgb

X_train = train_part[feature_cols]
y_train = train_part["sales_log"]

X_val = val_part[feature_cols]
y_val = val_part["sales_log"]

model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="rmse",
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(period=50)]
)

C:\Users\hyuhi\projects\store-sales-forecasting\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.088118 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2457
[LightGBM] [Info] Number of data points in the train set: 2972376, number of used features: 22
[LightGBM] [Info] Start training from score 2.919643
Training until validation scores don't improve for 50 rounds
[50]	valid_0's rmse: 0.454725	valid_0's l2: 0.206775
[100]	valid_0's rmse: 0.399548	valid_0's l2: 0.159638
[150]	valid_0's rmse: 0.396383	valid_0's l2: 0.157119
[200]	valid_0's rmse: 0.393884	valid_0's l2: 0.155145
[250]	valid_0's rmse: 0.391538	valid_0's l2: 0.153302
[300]	valid_0's rmse: 0.38944	valid_0's l2: 0.151663
[350]	valid_0's rmse: 0.388201	valid_0's l2: 0.1507
[400]	valid_0's rmse: 0.387053	valid_0's l2: 0.14981
[450]	valid_0's rmse: 0.385853	valid_0's l2: 0.148883
[500]	valid_0's rmse: 0.385799	valid_0's l2: 0.148

,learning_rate,0.05
,n_estimators,500
,random_state,42
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
